# 03_test_eval_qwen3_5_9b_lora

Strict span-level evaluation for the held-out `test_ground_truth.jsonl` generated by `01_build_sft_dataset.ipynb`.

This notebook loads the 9B base model plus LoRA adapter, generates tagged outputs, parses `<pii>` spans, and reports exact `type/start/end/value` micro and per-type metrics.


In [ ]:

import os
import json
import re
import time
from collections import Counter
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

from redaction_utils import parse_annotated


## 1. Paths and Settings

In [ ]:

BASE_MODEL = "../../model/Qwen3.5-9B-Base"
ADAPTER_DIR = "../outputs/qwen3_5_9b_base_lora_tagged_28_fastretry"
TEST_PATH = "../data/processed/test_ground_truth.jsonl"

OUT_PATH = os.path.join(ADAPTER_DIR, "test_eval_predictions_fast.jsonl")
SUMMARY_PATH = os.path.join(ADAPTER_DIR, "test_eval_summary_fast.json")

BATCH_SIZE = 16
MAX_INPUT_LEN = 1536
MAX_NEW_TOKENS = 512
MAX_EXAMPLES = None  # None = full test set; use 200 for a quick smoke test
PRINT_EVERY = 100

assert os.path.exists(BASE_MODEL), f"Base model not found: {BASE_MODEL}"
assert os.path.exists(ADAPTER_DIR), f"Adapter dir not found: {ADAPTER_DIR}"
assert os.path.exists(TEST_PATH), f"Test file not found: {TEST_PATH}"

print("BASE_MODEL  =", BASE_MODEL)
print("ADAPTER_DIR =", ADAPTER_DIR)
print("TEST_PATH   =", TEST_PATH)
print("OUT_PATH    =", OUT_PATH)
print("SUMMARY_PATH=", SUMMARY_PATH)
print("BATCH_SIZE  =", BATCH_SIZE)
print("MAX_INPUT_LEN=", MAX_INPUT_LEN)


## 2. Helpers

In [ ]:

def strip_think_blocks(text: str) -> str:
    return re.sub(r"<think>.*?</think>\s*", "", text, flags=re.DOTALL).strip()


def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                yield json.loads(line)


def span_key(s):
    return (s["type"], s["start"], s["end"], s["value"])


def prf(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * p * r / (p + r) if p + r else 0.0
    return p, r, f1


def make_system_prompt(target_labels):
    return (
        "You are a PII annotator for Australian context.\n"
        "Return the SAME text with supported PII wrapped as <pii type=\"TYPE\">VALUE</pii>.\n"
        "Preserve every character exactly. Do not paraphrase, summarize, or explain.\n"
        "Wrap every occurrence of supported PII. Do not deduplicate.\n"
        "If no supported PII is present, return the input unchanged.\n"
        "Supported types:\n- " + "\n- ".join(target_labels)
    )


## 3. Load Labels, Tokenizer, and Model

In [ ]:

with open(os.path.join(ADAPTER_DIR, "target_labels.json"), "r", encoding="utf-8") as f:
    target_labels = json.load(f)

SYSTEM_PROMPT = make_system_prompt(target_labels)
print("target label count:", len(target_labels))
print("target labels:", target_labels)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, trust_remote_code=True, use_fast=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

print("Loading base model...")
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)

print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(base, ADAPTER_DIR)
model.eval()

print("model loaded")
if torch.cuda.is_available():
    print(f"GPU allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")


## 4. Load Test Set

In [ ]:

test_items = list(load_jsonl(TEST_PATH))
if MAX_EXAMPLES is not None:
    test_items = test_items[:MAX_EXAMPLES]

print("test samples:", len(test_items))
print("first item keys:", list(test_items[0].keys()))
print("first labels:", test_items[0].get("labels", [])[:3])


## 5. Run Generation and Strict Span Evaluation

In [ ]:

Path(os.path.dirname(OUT_PATH)).mkdir(parents=True, exist_ok=True)

def build_prompt(text):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": text},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


@torch.inference_mode()
def infer_batch(texts):
    prompts = [build_prompt(t) for t in texts]
    tok_kwargs = {
        "return_tensors": "pt",
        "padding": True,
    }
    if MAX_INPUT_LEN is not None:
        tok_kwargs.update({"truncation": True, "max_length": MAX_INPUT_LEN})

    inputs = tokenizer(prompts, **tok_kwargs).to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    prompt_len = inputs["input_ids"].shape[1]
    decoded = []
    for row in outputs:
        generated_ids = row[prompt_len:]
        text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
        decoded.append(strip_think_blocks(text))
    return decoded


tp = Counter()
fp = Counter()
fn = Counter()
global_tp = global_fp = global_fn = 0
parse_fail = 0
roundtrip_fail = 0
unsupported_pred_types = Counter()
start_time = time.time()

print(
    f"Starting batched eval: samples={len(test_items)}, batch_size={BATCH_SIZE}, "
    f"max_input_len={MAX_INPUT_LEN}, max_new_tokens={MAX_NEW_TOKENS}",
    flush=True,
)

with open(OUT_PATH, "w", encoding="utf-8") as out_f:
    for batch_start in range(0, len(test_items), BATCH_SIZE):
        batch = test_items[batch_start:batch_start + BATCH_SIZE]
        texts = [item["text"] for item in batch]
        generated_batch = infer_batch(texts)

        for item, generated in zip(batch, generated_batch):
            text = item["text"]
            gold_spans = item.get("labels", [])

            pred_spans = []
            pred_plain = None
            err = None

            try:
                pred_plain, pred_spans = parse_annotated(generated, strict=False)
                if pred_plain != text:
                    roundtrip_fail += 1
            except Exception as e:
                parse_fail += 1
                err = repr(e)

            for s in pred_spans:
                if s.get("type") not in target_labels:
                    unsupported_pred_types[s.get("type")] += 1

            gold_set = {span_key(s) for s in gold_spans}
            pred_set = {span_key(s) for s in pred_spans if s.get("type") in target_labels}

            for k in gold_set & pred_set:
                tp[k[0]] += 1
                global_tp += 1
            for k in pred_set - gold_set:
                fp[k[0]] += 1
                global_fp += 1
            for k in gold_set - pred_set:
                fn[k[0]] += 1
                global_fn += 1

            out_f.write(json.dumps({
                "id": item.get("id"),
                "text": text,
                "gold_spans": gold_spans,
                "generated": generated,
                "pred_plain": pred_plain,
                "pred_spans": pred_spans,
                "parse_error": err,
                "roundtrip_ok": pred_plain == text,
                "tp": sorted(list(gold_set & pred_set)),
                "fp": sorted(list(pred_set - gold_set)),
                "fn": sorted(list(gold_set - pred_set)),
            }, ensure_ascii=False) + "\n")

        done = min(batch_start + len(batch), len(test_items))
        if done % PRINT_EVERY == 0 or done == len(test_items):
            p, r, f1 = prf(global_tp, global_fp, global_fn)
            elapsed = time.time() - start_time
            rate = done / max(elapsed, 0.01)
            eta = (len(test_items) - done) / max(rate, 0.01)
            print(
                f"{done}/{len(test_items)} | "
                f"P={p:.4f} R={r:.4f} F1={f1:.4f} | "
                f"parse_fail={parse_fail} roundtrip_fail={roundtrip_fail} | "
                f"{rate:.2f} samples/s | elapsed={elapsed/60:.1f} min | ETA={eta/60:.1f} min",
                flush=True,
            )

print("predictions saved:", OUT_PATH)


## 6. Summary

In [ ]:

per_type = {}
for t in target_labels:
    p, r, f1 = prf(tp[t], fp[t], fn[t])
    per_type[t] = {
        "tp": tp[t],
        "fp": fp[t],
        "fn": fn[t],
        "precision": p,
        "recall": r,
        "f1": f1,
    }

micro_p, micro_r, micro_f1 = prf(global_tp, global_fp, global_fn)
summary = {
    "n_samples": len(test_items),
    "batch_size": BATCH_SIZE,
    "max_input_len": MAX_INPUT_LEN,
    "max_new_tokens": MAX_NEW_TOKENS,
    "parse_fail": parse_fail,
    "roundtrip_fail": roundtrip_fail,
    "unsupported_pred_types": dict(unsupported_pred_types),
    "micro": {
        "tp": global_tp,
        "fp": global_fp,
        "fn": global_fn,
        "precision": micro_p,
        "recall": micro_r,
        "f1": micro_f1,
    },
    "per_type": per_type,
    "output_predictions": OUT_PATH,
    "elapsed_seconds": time.time() - start_time,
    "notes": [
        "Strict span-level match: type, start, end, and value must all match.",
        "This evaluates the held-out test_ground_truth.jsonl generated by 01.",
        "Fast version: batched generation, same strict span-level metrics.",
    ],
}

with open(SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("=== TEST EVAL SUMMARY ===")
print(json.dumps(summary["micro"], ensure_ascii=False, indent=2))
print("parse_fail:", parse_fail)
print("roundtrip_fail:", roundtrip_fail)
print("unsupported_pred_types:", dict(unsupported_pred_types))
print("summary saved:", SUMMARY_PATH)


## 7. Per-Type F1

In [ ]:

for t, v in summary["per_type"].items():
    print(
        f"{t:32s} "
        f"P={v['precision']:.4f} R={v['recall']:.4f} F1={v['f1']:.4f} "
        f"TP={v['tp']} FP={v['fp']} FN={v['fn']}"
    )
